In [32]:
!pip install -q pennylane==0.45.1

import time
import numpy as np
import pandas as pd
import torch
import pennylane as qml
from sklearn.metrics import roc_auc_score

seed = 0
np.random.seed(seed)
torch.manual_seed(seed)

In [33]:
data_dir = "/kaggle/input/datasets/malakkhaled202300369/nic-feature-dataset/prep"
n_qubits = 8   # first 8 pca components, one per qubit

angles, meta = {}, {}
for split in ["train", "val"]:
    angles[split] = np.load(f"{data_dir}/ffpp_{split}_angles32.npy").astype(np.float32)[:, :n_qubits]
    meta[split] = pd.read_csv(f"{data_dir}/ffpp_{split}_rows.csv")

is_inner = np.load(f"{data_dir}/ffpp_train_is_inner.npy").astype(bool)

x_train, x_val = angles["train"], angles["val"]
y_train = meta["train"]["label"].to_numpy().astype(np.float32)
y_val = meta["val"]["label"].to_numpy().astype(np.float32)

# inner fitness split and search-train both come from ff++ train
fit_idx = np.where(is_inner)[0]
search_idx = np.where(~is_inner)[0]

# the 4 fakes made from one pair share a video_id, so a video is (method, video_id)
fit_videos = meta["train"].iloc[fit_idx][["method", "video_id"]].drop_duplicates()
val_videos = meta["val"][["method", "video_id"]].drop_duplicates()

print("inner fitness frames:", len(fit_idx))
print("search-train frames:", len(search_idx))
print("inner fitness videos:", len(fit_videos))
print("val videos:", len(val_videos))
print("angle range:", x_train.min().round(3), "to", x_train.max().round(3))

inner fitness frames: 6912
search-train frames: 39164
inner fitness videos: 540
val videos: 700
angle range: 0.0 to 3.142


In [34]:
def build_circuit(n_layers, entangler):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(x, w):
        for i in range(n_qubits):
            qml.RY(x[:, i], wires=i)
        for layer in range(n_layers):
            for i in range(n_qubits):
                qml.Rot(w[layer, i, 0], w[layer, i, 1], w[layer, i, 2], wires=i)
            if entangler == "ring":
                for i in range(n_qubits):
                    qml.CNOT(wires=[i, (i + 1) % n_qubits])
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    return circuit


class QuantumClassifier(torch.nn.Module):
    def __init__(self, n_layers=2, entangler="ring"):
        super().__init__()
        self.circuit = build_circuit(n_layers, entangler)
        self.rot = torch.nn.Parameter(0.1 * torch.randn(n_layers, n_qubits, 3))
        self.readout = torch.nn.Linear(n_qubits, 1)

    def forward(self, x):
        out = torch.stack(self.circuit(x, self.rot), dim=1).float()
        return self.readout(out).squeeze(1)


def train(model, x, y, lr=0.01, epochs=5, batch_size=256, max_steps=None, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.as_tensor(x, dtype=torch.float32)
    y = torch.as_tensor(y, dtype=torch.float32)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    history = []
    step = 0
    for epoch in range(epochs):
        order = torch.randperm(len(x), generator=g)
        total = 0.0
        for start in range(0, len(x), batch_size):
            idx = order[start:start + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(x[idx]), y[idx])
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
            step += 1
            if max_steps is not None and step >= max_steps:
                break
        history.append(total / len(x))
        if max_steps is not None and step >= max_steps:
            break
    return history


def predict(model, x, batch_size=1000):
    out = []
    with torch.no_grad():
        for s in range(0, len(x), batch_size):
            xb = torch.as_tensor(x[s:s + batch_size], dtype=torch.float32)
            out.append(torch.sigmoid(model(xb)))
    return torch.cat(out).numpy()


def video_level(prob, y, info):
    df = info[["method", "video_id"]].copy()
    df["p"] = prob
    df["y"] = y
    g = df.groupby(["method", "video_id"]).agg(p=("p", "mean"), y=("y", "first"))
    return g["p"].to_numpy(), g["y"].to_numpy()


def bootstrap_ci(p, y, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    aucs = []
    while len(aucs) < n:
        idx = rng.integers(0, len(y), len(y))
        if y[idx].min() == y[idx].max():
            continue
        aucs.append(roc_auc_score(y[idx], p[idx]))
    return np.percentile(aucs, [2.5, 97.5])


torch.manual_seed(seed)
m = QuantumClassifier(n_layers=2, entangler="ring")
print("trainable weights:", sum(p.numel() for p in m.parameters()))

trainable weights: 57


In [35]:
rng = np.random.default_rng(1)


def make_xor(n):
    x = rng.uniform(0, np.pi, size=(n, n_qubits)).astype(np.float32)
    y = ((x[:, 0] > np.pi / 2) ^ (x[:, 1] > np.pi / 2)).astype(np.float32)
    return x, y


x_tr, y_tr = make_xor(2000)
x_te, y_te = make_xor(1000)

for entangler in ["ring", "none"]:
    torch.manual_seed(seed)
    model = QuantumClassifier(n_layers=2, entangler=entangler)
    train(model, x_tr, y_tr, lr=0.05, epochs=15, batch_size=128)
    auc = roc_auc_score(y_te, predict(model, x_te))
    print(f"entangler = {entangler:5s}   test auc = {auc:.3f}")

entangler = ring    test auc = 0.999
entangler = none    test auc = 0.465


In [36]:
runs = [
    ("ring, lr 0.01", "ring", 0.01),
    ("ring, lr 0.03", "ring", 0.03),
    ("ring, lr 0.1", "ring", 0.1),
    ("no entangler, lr 0.01", "none", 0.01),
]

table = []
for name, entangler, lr in runs:
    t0 = time.time()
    torch.manual_seed(seed)
    model = QuantumClassifier(n_layers=2, entangler=entangler)
    hist = train(model, x_train, y_train, lr=lr, epochs=5, batch_size=256, seed=seed)
    minutes = (time.time() - t0) / 60

    prob = predict(model, x_val)
    pv, yv = video_level(prob, y_val, meta["val"])
    lo, hi = bootstrap_ci(pv, yv)

    print(name, "| loss:", np.round(hist, 4), "| minutes:", round(minutes, 1), "| videos:", len(yv))
    table.append({"circuit": name,
                  "frame auc": round(roc_auc_score(y_val, prob), 3),
                  "video auc": round(roc_auc_score(yv, pv), 3),
                  "95% interval": f"{lo:.3f} to {hi:.3f}"})

print()
print(pd.DataFrame(table).to_string(index=False))

ring, lr 0.01 | loss: [0.6853 0.6715 0.6673 0.6661 0.6658] | minutes: 1.2 | videos: 700
ring, lr 0.03 | loss: [0.6759 0.6674 0.6649 0.6638 0.664 ] | minutes: 1.3 | videos: 700
ring, lr 0.1 | loss: [0.6748 0.6694 0.6693 0.6689 0.6679] | minutes: 1.2 | videos: 700
no entangler, lr 0.01 | loss: [0.6762 0.667  0.6667 0.666  0.6662] | minutes: 1.0 | videos: 700

              circuit  frame auc  video auc   95% interval
        ring, lr 0.01      0.630      0.644 0.594 to 0.691
        ring, lr 0.03      0.618      0.634 0.583 to 0.685
         ring, lr 0.1      0.622      0.640 0.592 to 0.687
no entangler, lr 0.01      0.618      0.643 0.592 to 0.692


In [37]:
# same 2000 frames as the grid and search notebooks
st = meta["train"].iloc[search_idx].copy()
st["row"] = search_idx
real_pick = st[st["label"] == 0].groupby("video_id").sample(n=2, random_state=0)["row"].to_numpy()
fake_pick = st[st["label"] == 1].groupby(["method", "video_id"]).sample(n=1, random_state=0)["row"].to_numpy()
rng = np.random.default_rng(seed)
sub = np.concatenate([rng.choice(real_pick, 1000, replace=False), rng.choice(fake_pick, 1000, replace=False)])


def fitness(n_layers, entangler, seed):
    t0 = time.time()
    torch.manual_seed(seed)
    model = QuantumClassifier(n_layers=n_layers, entangler=entangler)
    train(model, x_train[sub], y_train[sub], lr=0.03, epochs=6, batch_size=100, seed=seed)
    auc = roc_auc_score(y_train[fit_idx], predict(model, x_train[fit_idx]))
    return auc, time.time() - t0


genomes = [
    ("depth 1, no entangler", 1, "none"),
    ("depth 1, ring", 1, "ring"),
    ("depth 2, ring", 2, "ring"),
    ("depth 4, ring", 4, "ring")]

for name, n_layers, entangler in genomes:
    auc, secs = fitness(n_layers, entangler, seed)
    print(f"{name:22s} inner auc {auc:.3f} seconds per eval {secs:.1f}")

depth 1, no entangler  inner auc 0.627 seconds per eval 4.4
depth 1, ring          inner auc 0.524 seconds per eval 4.9
depth 2, ring          inner auc 0.610 seconds per eval 7.6
depth 4, ring          inner auc 0.627 seconds per eval 13.1
